In [ ]:
import sys
import os
import random
from pathlib import Path

import torch
from torch import optim
import torchvision
import numpy as np
from tqdm import tqdm
import matplotlib.pyplot as plt

In [ ]:
project_root = Path.cwd().parent 
sys.path.append(str(project_root))

In [ ]:
from img_generation.datasets import Config, DatasetType, load_dataloaders, VAE_TRANSFORM
from img_generation.models.vae import VQVAE, vqvae_loss, train_vq_vae, NUM_EMBEDDINGS
from img_generation.utils import EarlyStopping


In [ ]:
def set_seed(seed):
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.use_deterministic_algorithms(True)

if torch.cuda.is_available():
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(SEED)

# os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"  # wymagane przez niektóre operacje CUDA
device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.mps.is_available() else "cpu")
device

In [ ]:
config = Config(dataset_type=DatasetType.FASHION_MNIST, data_dir="../data/fashion_mnist", model_type="vae", batch_size=128, num_workers=4)
dataloaders, classes = load_dataloaders(config=config, transforms=VAE_TRANSFORM)

print(f"All classes: {classes}")
print(f"Number of all classes: {len(classes)}")

In [ ]:
for x, y in dataloaders["train"]:
    print(f"Input img shape: {x.shape}")
    print(f"class: {y}")
    break

In [ ]:
for x, _ in dataloaders["train"]:
    input_channels, input_size = x.shape[1], x.shape[2] # (B, C, H, W)
    break

In [ ]:
hidden_dim = 32
num_embeddings = NUM_EMBEDDINGS
# ^^Number of vectors in the codebook

vq_vae = VQVAE(in_channels=input_channels, hidden_dim=hidden_dim, num_embeddings=num_embeddings).to(device)
optimizer = optim.Adam(vq_vae.parameters(), lr=0.001)
scheduler = torch.optim.lr_scheduler.ExponentialLR(optimizer=optimizer, gamma=0.99)
early_stopping = EarlyStopping(patience=10, delta = 0.01)

In [ ]:
epochs = 70

early_stopping, val_lossses = train_vq_vae(model=vq_vae, loaders=dataloaders, criterion=vqvae_loss, optimizer=optimizer, lr_scheduler=scheduler, 
                                early_stopping=early_stopping, num_epochs=epochs, device=device)
early_stopping.load_best_model(vq_vae)

plt.plot(val_lossses)

In [ ]:
model_path = "../pretrained_models/vae"

torch.save(vq_vae.state_dict(), f=os.path.join(model_path, "vq_vae.pth"))

In [ ]:
def visualize_reconstructions(model, input_imgs, device):
    # Reconstruct images
    model.eval()
    with torch.no_grad():
        reconst_imgs, _ = model(input_imgs.to(device))
    reconst_imgs = reconst_imgs.cpu()
    
    # Plotting
    imgs = torch.stack([input_imgs, reconst_imgs], dim=1).flatten(0,1)
    grid = torchvision.utils.make_grid(imgs, nrow=6, normalize=False)
    grid = grid.permute(1, 2, 0)
    if len(input_imgs) == 6:
        plt.figure(figsize=(15,10))
    else:
        plt.figure(figsize=(15,15))
    plt.title("Reconstructions")
    plt.imshow(grid)
    plt.axis('off')
    plt.show()

In [ ]:
vq_vae.eval()
with torch.inference_mode():    
    test_losses = []
    for x, _ in tqdm(dataloaders["test"]):
        x = x.to(device)
        x_hat, loss = vq_vae(x)
        total_loss = vqvae_loss(x_hat, x, loss)
        test_losses.append(total_loss.item())
    print(f"Mean test loss: {np.mean(np.array(test_losses))}")

In [ ]:
n_samples = 27
sample_imgs = torch.stack([dataloaders["test"].dataset[i][0] for i in range(n_samples)], dim=0)    # wraps all tensors in one one external tensor - [[..], [..], .., [..]]
print(sample_imgs.shape)
visualize_reconstructions(vq_vae, input_imgs=sample_imgs, device=device)